# Acoustic-Aware Fusion — 4-class dysarthria severity (UA-Speech)

Three representations of the same utterance — **learned** (wav2vec 2.0), **segmental**
(MFCC+Δ+ΔΔ, F1–F3, HNR) and **suprasegmental** (F0, voicing, intensity, timing) — fused to
classify the severity of 15 dysarthric speakers (Very Low / Low / Mid / High intelligibility)
under **15-fold leave-one-speaker-out**.

| Section | What runs | Time |
|---|---|---|
| 1–3 | Environment, manifest, feature store for all 28 speakers | ~40 min first run, then cached |
| 4–5 | **Final model (AAF-Lite)**: frozen branches, control-referenced per word, late fusion with nested LOSO selection, every ablation, sanity checks | ~20 min first run |
| 6 | End-to-end LoRA gated-fusion network — kept for comparison, **off by default** | 3.5–6 h |

**Run All** gives the final result. Every step resumes from disk. `AAFA_SMOKE=1` runs a 4-fold
pipeline check instead (not a result).

## 1. Environment

In [ ]:
import os
import shutil
import sys
import time
from pathlib import Path

SESSION_START = time.monotonic()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch

# Started from notebooks/ or the repo root: find the directory holding src/.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src import feature_store
from src.aaflite.run import FINAL_TAG, SUMMARY_PATH, prepare_features, run_experiments
from src.console import print_header, print_kv, print_note, print_status, silence_library_progress
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model, parameter_counts
from src.training.reporting import (check_frozen_config_guard, collect_run_results, per_class_table,
                                    print_feature_audit, print_final_run_configuration,
                                    write_frozen_config)
from src.training.runner import TrainingConfig, build_folds, run_training
from src.training.session import calibrate_throughput, project_runtime, subset_blocks
from src.training.utils import (configure_local_runtime, gpu_temperature, memory_status,
                                on_battery, resolve_amp_dtype, resolve_device)
from src.vad_cache import verify_vad_span_cache

config.ensure_directories()
silence_library_progress()
# src.training.reporting selects the file-only Agg backend at import; display inline here.
%matplotlib inline
print_status(f"Project root: {PROJECT_ROOT}", ok=True)

In [ ]:
device = resolve_device(None)
if device.type != "cuda":
    raise RuntimeError("CUDA is not available to this kernel — select the Python environment with "
                       "the CUDA build of torch (the 'torch-gpu' conda env).")
configure_local_runtime(device)          # cuDNN autotuning + a VRAM cap sized to free GPU memory

props = torch.cuda.get_device_properties(device)
gpu_free, gpu_total = torch.cuda.mem_get_info(device)
memory = memory_status()
print_header("Environment")
print_kv("Python / PyTorch / CUDA", f"{sys.version.split()[0]} / {torch.__version__} / {torch.version.cuda}")
print_kv("GPU", f"{props.name}, {gpu_total / 2**30:.1f} GiB ({gpu_free / 2**30:.1f} GiB free), "
                f"{gpu_temperature()} °C")
print_kv("CPU", f"{psutil.cpu_count(logical=False)} cores / {psutil.cpu_count()} threads")
print_kv("RAM / commit free", f"{memory['ram_available']:.1f} of {memory['ram_total']:.1f} GiB / "
                              f"{memory['commit_available']:.1f} of {memory['commit_limit']:.1f} GiB")
print_kv("Disk free", f"{shutil.disk_usage(PROJECT_ROOT).free / 2**30:.0f} GiB")
print_kv("Power", "battery — training will wait for the charger" if on_battery() else "plugged in")
print_kv("Precision", str(resolve_amp_dtype(device)).replace("torch.", "") + " autocast")
print_kv("Thermal guard", f"pause at {config.GPU_TEMP_PAUSE_C} °C, resume at {config.GPU_TEMP_RESUME_C} °C")
# A fold commits ~10 GiB (on Windows the GPU memory PyTorch reserves counts too)
# plus ~2 GiB for its data-loading worker.
if memory["ram_available"] < 4 or memory["commit_available"] < 14:
    print_note("Little memory is free — close browsers and other heavy applications before training.")

## 2. Data
UA-Speech, microphone M6: 15 dysarthric speakers (the evaluation set) and 13 healthy controls (used only as the per-word reference of section 4 — never trained on, never tested).

In [ ]:
from src.extraction import extract_all_archives
from src.scanning import scan_audio_files, verify_speakers

EXPECTED_M6 = len(config.ALL_SPEAKERS) * config.WORDS_PER_SPEAKER          # 28 x 765
present = sum(1 for _ in config.AUDIO_DIR.rglob(f"*_{config.TARGET_MIC}.wav")) if config.AUDIO_DIR.exists() else 0
if present >= EXPECTED_M6:
    print_status(f"{present:,} {config.TARGET_MIC} files present in {config.AUDIO_DIR}.", ok=True)
else:
    missing = [str(config.ARCHIVE_DIR / name) for name in config.ARCHIVE_FILES
               if not (config.ARCHIVE_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f"Only {present:,}/{EXPECTED_M6:,} M6 files extracted and the archives "
                                f"are missing: {missing}")
    extract_all_archives()
    verify_speakers(scan_audio_files())

In [ ]:
df_full = load_manifest()          # outputs/m6_manifest.csv (regenerated from the audio if absent)
missing_files = int((~df_full["Filepath"].map(os.path.exists)).sum())
if missing_files:
    raise FileNotFoundError(f"{missing_files:,} manifest files are missing — delete "
                            f"{config.MANIFEST_PATH} to regenerate it.")
print_header("Manifest")
print_kv("Utterances (M6) / speakers", f"{len(df_full):,} / {df_full['Speaker_ID'].nunique()}")
composition = df_full.groupby(["Severity", "Block"]).size().unstack(fill_value=0)
composition["total"] = composition.sum(axis=1)
display(composition)

## 3. Feature store
VAD spans and the frame-wise segmental (43 × 401) and suprasegmental (3 × 401) features of every
utterance, for **all 28 speakers**, in resumable per-(speaker, block) chunks. Sampled entries are
re-computed live and must match bit for bit.

In [ ]:
feature_store.build_feature_store(df_full, n_workers=config.FEATURE_STORE_WORKERS)
coverage = feature_store.feature_store_coverage(df_full)
if not coverage["complete"]:
    raise RuntimeError(f"Feature store incomplete (missing {coverage['chunks_missing']}) — re-run this cell.")
verify_vad_span_cache(df_full, n=300)
feature_store.verify_feature_store(df_full, n=24)

## 4. Final model — AAF-Lite

**Branches** (all frozen — nothing is fitted by gradient descent):
* **learned** — every wav2vec 2.0 hidden state pooled over real frames into mean + std, averaged
  within a layer group (early 1–4, middle 5–8, late 9–12, all 1–12);
* **segmental / suprasegmental** — mean, std, p10, p50, p90 over valid frames (F0 over voiced
  frames), plus true speech duration, voiced ratio and F0 range.

**Control reference.** Every speaker reads the same 765 prompts. Each feature is expressed relative
to the healthy controls' mean for the *same prompt* (block × word), scaled by their pooled
within-prompt spread, plus the RMS distance from healthy speech. This removes what was said and
keeps how it was said. It uses no labels and no test speaker.

**Nested leave-one-speaker-out.** Per outer fold (one held-out speaker), an inner LOSO over the
14 training speakers chooses each branch's layer group, PCA size and L2 strength, then the
late-fusion weights. The held-out speaker never influences a choice.

In [ ]:
SMOKE_TEST = os.environ.get("AAFA_SMOKE") == "1"
FOLDS = ["F03", "F02", "F04", "M08"] if SMOKE_TEST else None   # one speaker per class
if SMOKE_TEST:
    print_note("SMOKE TEST — 4 outer folds, pipeline check only, not a result.")

df_dys, features = prepare_features(df_full, device=device)
pd.DataFrame([{"Branch / variant": key, "Dims (referenced)": X.shape[1],
               "Dims (raw)": features["raw"][key].shape[1]} for key, X in features["ref"].items()])

Final model, every branch subset, the fusion without the control reference, the majority baseline, and a **permuted-label sanity check** (severity labels shuffled across speakers — a leak-free protocol must fall to chance on it).

In [ ]:
start = time.monotonic()
summary, outputs = run_experiments(df_dys, features, folds=FOLDS)
print_kv("Nested LOSO wall clock", f"{(time.monotonic() - start) / 60:.1f} min")
print_kv("Summary", SUMMARY_PATH.relative_to(PROJECT_ROOT))
(summary.style.format({c: "{:.3f}" for c in ("Accuracy", "Macro F1", "Balanced acc.", "Ordinal MAE", "AUROC")},
                      na_rep="N/A")
 .background_gradient(subset=["Accuracy"], cmap="RdYlGn", vmin=0.25, vmax=0.75)
 .hide(axis="index").set_caption("All runs — pooled over held-out speakers"))

## 5. Results — final model (control-referenced three-branch fusion)

In [ ]:
final = outputs[FINAL_TAG]
p = final["pooled"]
print_header("Final model")
print_kv("Utterance accuracy", f"{p['accuracy']:.3f}  (95% CI over speakers "
                               f"{p['accuracy_ci95'][0]:.3f}-{p['accuracy_ci95'][1]:.3f})")
print_kv("Macro F1 / balanced accuracy", f"{p['f1']:.3f} / {p['balanced_accuracy']:.3f}")
print_kv("Ordinal MAE / AUROC", f"{p['ordinal_mae']:.3f} / {p['auroc']:.3f}")
print_kv("Speakers correct", f"{p['speakers_correct']}/{p['n_speakers']}  (95% CI "
                             f"{p['speaker_accuracy_ci95'][0]:.2f}-{p['speaker_accuracy_ci95'][1]:.2f})")
preds = final["predictions"]
display(per_class_table(preds["y_true"].to_numpy(), preds["y_pred"].to_numpy(), final["y_prob"])
        .style.format({c: "{:.3f}" for c in ("precision", "recall", "f1", "auroc_ovr")}, na_rep="N/A")
        .hide(axis="index").set_caption("Per class"))
display(final["speakers"].style.format({"Utterance accuracy": "{:.3f}"}).hide(axis="index")
        .set_caption("Speaker level — median of each speaker's utterance predictions"))

In [ ]:
names = config.SEVERITY_CLASS_NAMES
matrix = (pd.crosstab(preds["y_true"], preds["y_pred"])
          .reindex(index=range(4), columns=range(4), fill_value=0).to_numpy())
share = matrix / np.clip(matrix.sum(axis=1, keepdims=True), 1, None)
per_fold = pd.DataFrame(final["folds"])
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4.6))
image = left.imshow(share, cmap="Blues", vmin=0, vmax=1)
left.set_xticks(range(4), names, rotation=30, ha="right")
left.set_yticks(range(4), names)
left.set_xlabel("Predicted severity")
left.set_ylabel("True severity")
left.set_title("Pooled confusion — final model")
for i in range(4):
    for j in range(4):
        left.text(j, i, f"{matrix[i, j]:,}\n{share[i, j]:.0%}", ha="center", va="center",
                  fontsize=9, color="white" if share[i, j] > 0.5 else "black")
figure.colorbar(image, ax=left, shrink=0.8, label="share of true class")
ordered = per_fold.assign(order=per_fold["true_label"].map(names.index)).sort_values(["order", "fold"])
right.bar(ordered["fold"], ordered["accuracy"], color=plt.cm.viridis(ordered["order"] / 3))
for index, name in enumerate(names):
    right.bar(0, 0, color=plt.cm.viridis(index / 3), label=name)
right.set_ylim(0, 1)
right.set_ylabel("Held-out accuracy")
right.set_title("Per held-out speaker (colour = true severity)")
right.legend(loc="upper right", fontsize=8)
figure.tight_layout()
plt.show()

What the inner loop chose in each outer fold — configurations, fusion weights and decision rule, selected on the 14 training speakers only.

In [ ]:
pd.DataFrame([{"Held-out": r["fold"], "True": r["true_label"], "Accuracy": r["accuracy"],
               **{f"w_{b}": w for b, w in r["weights"].items()}, "decoder": r["decoder"],
               "learned layers": r["configs"]["learned"]["variant"],
               "learned PCA / C": f"{r['configs']['learned']['pca_dims']} / {r['configs']['learned']['C']}",
               "segmental PCA / C": f"{r['configs']['segmental']['pca_dims']} / {r['configs']['segmental']['C']}",
               "supra C": r["configs"]["supra"]["C"]} for r in final["folds"]]
             ).style.format({"Accuracy": "{:.3f}", "w_learned": "{:.1f}", "w_segmental": "{:.1f}",
                             "w_supra": "{:.1f}"}).hide(axis="index")

## 6. End-to-end model (comparison, off by default)

The LoRA-finetuned gated-fusion network (`src/models/gated_fusion.py`): wav2vec 2.0 + LoRA,
segmental and suprasegmental CNNs, softmax gate, CORAL ordinal head, adversarial speaker head —
trained per fold under the same 15-fold protocol. Two complete runs took 3.4 h and 6.0 h; their
per-fold results are in `docs/end_to_end_v2_log.txt`. Set `RUN_END_TO_END = True` to re-train it.

In [ ]:
RUN_END_TO_END = False

end_to_end = pd.DataFrame([
    {"Model": "End-to-end gated fusion v1 (LoRA)", "Accuracy": 0.453, "Macro F1": 0.312,
     "Balanced acc.": 0.378, "Ordinal MAE": 0.998, "Speakers correct": "8/15"},
    {"Model": "End-to-end gated fusion v2 (LoRA)", "Accuracy": 0.370, "Macro F1": 0.296,
     "Balanced acc.": 0.318, "Ordinal MAE": 1.132, "Speakers correct": "6/15"},
])
ours = summary[summary["Model"].isin([FINAL_TAG, "fusion_raw", "majority_baseline"])][end_to_end.columns]
(pd.concat([ours, end_to_end], ignore_index=True).style
 .format({c: "{:.3f}" for c in ("Accuracy", "Macro F1", "Balanced acc.", "Ordinal MAE")}, na_rep="N/A")
 .hide(axis="index").set_caption("Final model vs. the end-to-end network (same 15-fold protocol)"))

### 6.1 Run parameters
`MODEL` selects the full network or one of eight ablations; each gets its own `RUN_NAME`.

In [ ]:
if RUN_END_TO_END:
    MODEL = SEVERITY_MODEL_NAME              # or: ab1_wav2vec2_only, ab2_acoustic_only, ab3_wav2vec2_acoustic_concat,
                                             #     ab4_wav2vec2_segmental, ab5_wav2vec2_suprasegmental,
                                             #     ab6_full_fusion, ab7_full_complementarity, ab8_full_speaker_grl
    RUN_BLOCKS = ["B1", "B2", "B3"]          # all 765 utterances per speaker
    EPOCHS, PATIENCE = config.DEFAULT_EPOCHS, config.DEFAULT_PATIENCE
    BATCH_SIZE = config.DEFAULT_BATCH_SIZE
    GRADIENT_CHECKPOINTING = config.WAV2VEC_GRADIENT_CHECKPOINTING
    SESSION_HOURS = None                     # e.g. 8: stop cleanly before 8 h; re-run to continue
    RUN_BENCHMARK = False                    # True: re-measure batch size x checkpointing (~7 min)
    SMOKE_TEST = os.environ.get("AAFA_SMOKE") == "1"

    RUN_TAG = "v2"                          # new protocol (min epochs, smoothed monitor, 2 val speakers/class): do not mix with v1 folds
    RUN_NAME = f"sev_{MODEL}_{'+'.join(RUN_BLOCKS)}_{RUN_TAG}" + ("_smoke" if SMOKE_TEST else "")
    df_run = subset_blocks(df_full, RUN_BLOCKS)
    df_dysarthric = df_run[df_run["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].reset_index(drop=True)

    # Every fold must be present and complete: a missing speaker would silently
    # drop a LOSO fold, a short one would bias its test set.
    per_speaker = df_dysarthric.groupby("Speaker_ID").size()
    expected_per_speaker = len(RUN_BLOCKS) * config.WORDS_PER_SPEAKER // 3
    assert sorted(per_speaker.index) == sorted(config.DYSARTHRIC_IDS), "a dysarthric speaker is missing"
    assert (per_speaker == expected_per_speaker).all(), per_speaker[per_speaker != expected_per_speaker]
    fold_ids = [fold_id for fold_id, _, _ in build_folds(df_run)]

    print_kv("Run name", RUN_NAME)
    print_kv("Severity utterances", f"{len(df_dysarthric):,} = 15 speakers x {expected_per_speaker}")
    print_kv("LOSO folds", f"{len(fold_ids)}: {', '.join(fold_ids)}")
    print_kv("Speakers per class", ", ".join(f"{c} {sum(config.SEVERITY_MAP[s] == c for s in config.DYSARTHRIC_IDS)}"
                                              for c in config.SEVERITY_CLASS_NAMES))
    if SMOKE_TEST:
        print_note("SMOKE TEST — pipeline check only, not a result.")

### 6.2 Model and one real batch

In [ ]:
if RUN_END_TO_END:
    from torch.utils.data import DataLoader
    from src.dataset import UASpeechDataset

    batch = next(iter(DataLoader(UASpeechDataset(df_dysarthric.sample(n=4, random_state=config.DEFAULT_SEED)),
                                 batch_size=4)))
    assert batch["segmental"].shape[1:] == (config.SEGMENTAL_CHANNELS, 401)
    assert batch["supra"].shape[1:] == (config.SUPRA_CHANNELS, 401)
    assert batch["waveform"].shape[-1] == config.MAX_SAMPLES
    print_status("A real batch matches the model's input contract "
                 + ", ".join(f"{k} {tuple(batch[k].shape)}" for k in ("waveform", "segmental", "supra")), ok=True)

    model_preview = build_model(MODEL, num_speakers=11, gradient_checkpointing=GRADIENT_CHECKPOINTING)
    counts = parameter_counts(model_preview)
    print_header(f"Model — {MODEL}")
    print_kv("Trainable parameters", f"{counts['trainable_params']:,} of {counts['total_params']:,} "
                                     f"({counts['trainable_pct']:.2f}%): LoRA adapters, branches, heads")
    print_feature_audit(model_preview)
    del model_preview
    torch.cuda.empty_cache()

### 6.3 Throughput and projected run time

In [ ]:
if RUN_END_TO_END:
    if RUN_BENCHMARK:
        from src.training.session import benchmark_batch_sizes
        measurement, _ = benchmark_batch_sizes(df_run, model_name=MODEL, batch_sizes=(32, 64))
        BATCH_SIZE, GRADIENT_CHECKPOINTING = measurement.batch_size, measurement.gradient_checkpointing
    else:
        measurement = calibrate_throughput(df_run, model_name=MODEL, batch_size=BATCH_SIZE,
                                           gradient_checkpointing=GRADIENT_CHECKPOINTING)
    projection = project_runtime(df_run, measurement, epochs=EPOCHS)
    print_header("Projected run time")
    print_kv("Mean fold (train / val / test)", f"{projection['n_train']} / {projection['n_val']} / {projection['n_test']}")
    print_kv("Epoch / fold at max epochs", f"{projection['epoch_seconds'] / 60:.1f} min / {projection['fold_seconds'] / 60:.0f} min")
    print_kv(f"{projection['n_folds']} folds, upper bound", f"{projection['total_seconds'] / 3600:.1f} h")

### 6.4 Training
The configuration is frozen under `outputs/results/<RUN_NAME>/`; re-running the same `RUN_NAME` with a different configuration is refused. Interrupting is safe: re-running resumes.

In [ ]:
if RUN_END_TO_END:
    cfg = TrainingConfig(
        model=MODEL, run_name=RUN_NAME,
        epochs=1 if SMOKE_TEST else EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        session_hours=SESSION_HOURS, fold_time_estimate_s=projection["fold_seconds"],
        max_folds=2 if SMOKE_TEST else None, limit_samples=64 if SMOKE_TEST else None,
    )
    final_config = print_final_run_configuration(cfg)
    check_frozen_config_guard(final_config)
    print_kv("Frozen to", write_frozen_config(final_config).relative_to(PROJECT_ROOT))

In [ ]:
if RUN_END_TO_END:
    training_start = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    e2e_summary, e2e_pooled = run_training(df_run, cfg)
    print_kv("Training wall clock", f"{(time.monotonic() - training_start) / 3600:.2f} h")
    print_kv("Peak GPU memory", f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

### 6.5 End-to-end results
Rebuilt from the files on disk.

In [ ]:
if RUN_END_TO_END:
    from IPython.display import Markdown

    expected_folds = fold_ids[:cfg.max_folds] if cfg.max_folds else fold_ids
    results = collect_run_results(RUN_NAME, expected_folds)
    cov = results["coverage"]
    display(Markdown(
        f"### Run status: **{cov['status']}** — {cov['completed']} / {cov['expected']} folds ({cov['completion_pct']:.0f}%)\n"
        + (f"- Missing: {', '.join(cov['missing_folds'])}\n" if cov["missing_folds"] else "")
        + (f"- Failed: {', '.join(cov['failed_folds'])}\n" if cov["failed_folds"] else "")
        + ("\n> **Partial — not a final result.** Re-run the training cell to finish the missing folds.\n"
           if cov["status"] != "COMPLETE" else "")))

    per_fold = results["per_fold"]
    if per_fold.empty:
        display(Markdown("_No fold has finished yet._"))
    else:
        display(per_fold.style
                .format({"Accuracy": "{:.3f}", "Ordinal MAE": "{:.3f}", "Train (min)": "{:.1f}"}, na_rep="N/A")
                .background_gradient(subset=["Accuracy"], cmap="RdYlGn", vmin=0, vmax=1)
                .background_gradient(subset=["Ordinal MAE"], cmap="RdYlGn_r", vmin=0, vmax=3)
                .hide(axis="index").set_caption("Per held-out speaker"))

In [ ]:
if RUN_END_TO_END:
    if "pooled" in results:
        tag = "" if cov["status"] == "COMPLETE" else f" — {cov['status']}, NOT A FINAL RESULT"
        display(Markdown(f"### Pooled over {cov['completed']} held-out speakers ({results['n_utterances']:,} utterances){tag}"))
        display(results["pooled"].style.format({"Value": "{:.4f}"}, na_rep="N/A").hide(axis="index"))
        display(results["per_class"].style.format(
            {c: "{:.3f}" for c in ("precision", "recall", "f1", "auroc_ovr")}, na_rep="N/A")
            .hide(axis="index").set_caption("Per class (N/A: never predicted, or absent from the pooled set)"))
        display(results["speakers"].style.format({"Utterance accuracy": "{:.3f}"}).hide(axis="index")
                .set_caption("Speaker level — median of each speaker's utterance predictions"))

In [ ]:
if RUN_END_TO_END:
    if "confusion" in results:
        names = config.SEVERITY_CLASS_NAMES
        matrix = results["confusion"].to_numpy()
        share = matrix / np.clip(matrix.sum(axis=1, keepdims=True), 1, None)
        figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4.6))
        image = left.imshow(share, cmap="Blues", vmin=0, vmax=1)
        left.set_xticks(range(4), names, rotation=30, ha="right")
        left.set_yticks(range(4), names)
        left.set_xlabel("Predicted severity")
        left.set_ylabel("True severity")
        left.set_title(f"Pooled confusion — {cov['completed']}/{cov['expected']} folds")
        for i in range(4):
            for j in range(4):
                left.text(j, i, f"{matrix[i, j]:,}\n{share[i, j]:.0%}", ha="center", va="center",
                          fontsize=9, color="white" if share[i, j] > 0.5 else "black")
        figure.colorbar(image, ax=left, shrink=0.8, label="share of true class")

        ordered = per_fold.assign(order=per_fold["True class"].map(names.index)).sort_values(["order", "Fold"])
        right.bar(ordered["Fold"], ordered["Accuracy"], color=plt.cm.viridis(ordered["order"] / 3))
        for index, name in enumerate(names):
            right.bar(0, 0, color=plt.cm.viridis(index / 3), label=name)
        right.set_ylim(0, 1)
        right.set_ylabel("Held-out accuracy")
        right.set_title("Per held-out speaker (colour = true severity)")
        right.legend(loc="upper right", fontsize=8)
        figure.tight_layout()
        plt.show()

In [ ]:
print_kv("Artifacts", config.OUTPUT_DIR)
print_kv("Whole notebook", f"{(time.monotonic() - SESSION_START) / 60:.1f} min")